# Pre-Report Inspector (Docling + LLM Summary)

Use this notebook to inspect what was produced before final PDF/report assembly:
- `full_text` (Docling markdown extraction)
- `summary` (per-paper LLM summary)

This reads data from `data/queries/<query_id>.json`.

In [ ]:
from pathlib import Path
import json
from IPython.display import Markdown, display

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent

queries_dir = repo_root / "data" / "queries"
queries_dir

In [ ]:
# Optional: set a specific query_id (without .json).
# Example: query_id = "a1b2c3d4"
query_id = None

def load_query_state(queries_dir: Path, query_id: str | None = None):
    if query_id:
        target = queries_dir / f"{query_id}.json"
        if not target.exists():
            raise FileNotFoundError(f"Query file not found: {target}")
        return target, json.loads(target.read_text(encoding="utf-8"))

    files = sorted(queries_dir.glob("*.json"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not files:
        raise FileNotFoundError(f"No query files found in {queries_dir}")

    # Prefer the newest run that has papers.
    for fp in files:
        try:
            data = json.loads(fp.read_text(encoding="utf-8"))
        except Exception:
            continue
        if data.get("papers"):
            return fp, data

    # Fallback to newest file even if papers are empty.
    fp = files[0]
    return fp, json.loads(fp.read_text(encoding="utf-8"))

state_path, state = load_query_state(queries_dir, query_id=query_id)
print(f"Loaded: {state_path.name}")
print(f"Query: {state.get('query', '<missing>')}")
print(f"Status: {state.get('status', '<missing>')}")

In [ ]:
papers = state.get("papers", [])
print(f"Papers found: {len(papers)}")

for i, p in enumerate(papers, start=1):
    m = p.get("metadata", {})
    title = (m.get("title") or "Untitled").replace("\n", " ")
    print(f"{i:>2}. [{m.get('source', '?')}] {m.get('published_date', '?')} | {title}")

In [ ]:
def show_paper(idx: int, full_text_chars: int = 6000):
    if idx < 1 or idx > len(papers):
        raise IndexError(f"idx must be between 1 and {len(papers)}")

    p = papers[idx - 1]
    m = p.get("metadata", {})

    title = m.get("title", "Untitled")
    source = m.get("source", "")
    published = m.get("published_date", "")
    url = m.get("url", "")

    summary = p.get("summary") or "_No LLM summary stored for this paper._"
    full_text = p.get("full_text") or ""

    display(Markdown(f"## Paper {idx}: {title}"))
    display(Markdown(f"**Source:** {source}  \n**Published:** {published}  \n**URL:** {url}"))

    display(Markdown("### LLM Summary (pre-report)"))
    display(Markdown(summary))

    snippet = full_text[:full_text_chars]
    if len(full_text) > full_text_chars:
        snippet += "\n\n... (truncated) ..."

    display(Markdown("### Docling Markdown (raw, pre-report)"))
    display(Markdown(f"```markdown\n{snippet}\n```"))

# Example:
show_paper(1)